# Indexa Capital — Análisis de Aportaciones y Crecimiento
Datos diarios desde **2015-12-31** hasta **2026-04-09**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

def parse_eu_number(s):
    """Convierte formato europeo '1.064.211,00' a float."""
    return float(str(s).replace('.', '').replace(',', '.'))

def fmt_eur(x, pos=None):
    if abs(x) >= 1e9:
        return f'{x/1e9:.1f}B €'
    elif abs(x) >= 1e6:
        return f'{x/1e6:.0f}M €'
    elif abs(x) >= 1e3:
        return f'{x/1e3:.0f}K €'
    return f'{x:.0f} €'

## 1. Carga de datos

In [ ]:
def load_csv(path, cols):
    df = pd.read_csv(path, sep=';', header=0, names=['fecha'] + cols,
                     parse_dates=['fecha'], dayfirst=False)
    for col in cols:
        df[col] = df[col].apply(parse_eu_number)
    df = df.set_index('fecha').sort_index()
    return df

rev = load_csv('indexa_stats_revenue.csv',
               ['volumen', 'ingresos_dia', 'ingresos_anual', 'comision_pct'])

vol = load_csv('indexa_stats_volume.csv',
               ['volumen', 'aportaciones_dia', 'aportaciones_acum'])

# Merge en un único DataFrame
df = vol.join(rev[['ingresos_dia', 'ingresos_anual', 'comision_pct']], how='left')
df['año'] = df.index.year
df['mes'] = df.index.month
df['año_mes'] = df.index.to_period('M')

print(f"Rango de datos: {df.index.min().date()} → {df.index.max().date()}")
print(f"Total días: {len(df):,}")
df.tail(3)

## 2. Resumen general

In [ ]:
volumen_inicio = df['volumen'].iloc[0]
volumen_final  = df['volumen'].iloc[-1]
aportaciones_total = df['aportaciones_acum'].iloc[-1]
rentabilidad_total = volumen_final - aportaciones_total
crecimiento_volumen = (volumen_final / volumen_inicio - 1) * 100
ingresos_anuales_actual = df['ingresos_anual'].iloc[-1]
comision_actual = df['comision_pct'].iloc[-1]

print("══════════════════════════════════════════════")
print(" RESUMEN INDEXA CAPITAL")
print("══════════════════════════════════════════════")
print(f" Volumen inicial (2015-12-31):  {volumen_inicio:>18,.0f} €")
print(f" Volumen actual:                {volumen_final:>18,.0f} €")
print(f" Crecimiento total volumen:     {crecimiento_volumen:>17.1f} %")
print("──────────────────────────────────────────────")
print(f" Aportaciones netas acumuladas: {aportaciones_total:>18,.0f} €")
print(f" Rentabilidad generada:         {rentabilidad_total:>18,.0f} €")
print(f" % Rentabilidad s/aportaciones: {rentabilidad_total/aportaciones_total*100:>17.1f} %")
print("──────────────────────────────────────────────")
print(f" Ingresos anuales recurrentes:  {ingresos_anuales_actual:>18,.0f} €")
print(f" Comisión media actual:         {comision_actual:>17.3f} %")
print("══════════════════════════════════════════════")

## 3. Aportaciones por año

In [ ]:
anual = df.groupby('año').agg(
    aportaciones_netas=('aportaciones_dia', 'sum'),
    dias_con_aportacion=('aportaciones_dia', lambda x: (x > 0).sum()),
    dias_con_retirada=('aportaciones_dia', lambda x: (x < 0).sum()),
    aportacion_media_dia=('aportaciones_dia', lambda x: x[x > 0].mean()),
    aportacion_max_dia=('aportaciones_dia', 'max'),
    volumen_fin=('volumen', 'last'),
    ingresos_anuales=('ingresos_anual', 'last'),
).round(0)

anual['crecimiento_volumen_pct'] = anual['volumen_fin'].pct_change() * 100
anual['crecimiento_aport_pct']   = anual['aportaciones_netas'].pct_change() * 100

print(anual[['aportaciones_netas', 'dias_con_aportacion', 'aportacion_media_dia',
             'aportacion_max_dia', 'volumen_fin', 'crecimiento_volumen_pct']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Aportaciones netas por año
años = anual.index.astype(str)
colores = ['#2196F3' if v >= 0 else '#F44336' for v in anual['aportaciones_netas']]
bars = axes[0].bar(años, anual['aportaciones_netas'] / 1e6, color=colores, width=0.6)
axes[0].set_title('Aportaciones netas anuales', fontweight='bold')
axes[0].set_ylabel('Millones €')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
for bar, val in zip(bars, anual['aportaciones_netas']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val/1e6:.0f}M', ha='center', va='bottom', fontsize=8, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Volumen total al cierre de cada año
axes[1].bar(años, anual['volumen_fin'] / 1e9, color='#4CAF50', width=0.6)
axes[1].set_title('Volumen gestionado al cierre de año', fontweight='bold')
axes[1].set_ylabel('Miles de millones €')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}B'))
for i, (idx, row) in enumerate(anual.iterrows()):
    axes[1].text(i, row['volumen_fin']/1e9 + 0.02,
                 f'{row["volumen_fin"]/1e9:.2f}B', ha='center', va='bottom', fontsize=8, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Evolución del volumen: aportaciones vs rentabilidad

In [ ]:
# Resamplear a frecuencia mensual (último valor del mes)
monthly = df.resample('ME').last()
monthly['rentabilidad_acum'] = monthly['volumen'] - monthly['aportaciones_acum']

fig, ax = plt.subplots(figsize=(16, 6))
ax.stackplot(monthly.index,
             monthly['aportaciones_acum'] / 1e9,
             monthly['rentabilidad_acum'].clip(lower=0) / 1e9,
             labels=['Aportaciones netas acumuladas', 'Rentabilidad generada'],
             colors=['#2196F3', '#4CAF50'], alpha=0.85)
ax.set_title('Volumen gestionado: Aportaciones vs Rentabilidad (acumulado mensual)', fontweight='bold', fontsize=13)
ax.set_ylabel('Miles de millones €')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}B'))
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 5. Distribución mensual de aportaciones

In [ ]:
mensual = df.groupby('año_mes')['aportaciones_dia'].sum().reset_index()
mensual['año_mes_dt'] = mensual['año_mes'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(18, 5))
colores = ['#2196F3' if v >= 0 else '#F44336' for v in mensual['aportaciones_dia']]
ax.bar(mensual['año_mes_dt'], mensual['aportaciones_dia'] / 1e6, color=colores, width=25)
ax.set_title('Aportaciones netas mensuales', fontweight='bold', fontsize=13)
ax.set_ylabel('Millones €')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax.axhline(0, color='black', linewidth=0.7)
plt.tight_layout()
plt.show()

## 6. Estacionalidad: aportaciones por mes del año

In [ ]:
meses_nombre = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

por_mes = df.groupby('mes')['aportaciones_dia'].sum() / df.groupby('mes')['año'].nunique()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(meses_nombre, por_mes.values / 1e6, color='#7C4DFF', width=0.6)
ax.set_title('Aportaciones netas medias por mes (media de todos los años)', fontweight='bold', fontsize=13)
ax.set_ylabel('Millones €')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
for bar, val in zip(bars, por_mes.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val/1e6:.0f}M', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 7. Ingresos recurrentes anuales (ARR) y comisión media

In [ ]:
monthly_arr = df.resample('ME').last()[['ingresos_anual', 'comision_pct']]

fig, ax1 = plt.subplots(figsize=(16, 5))
ax2 = ax1.twinx()

ax1.fill_between(monthly_arr.index, monthly_arr['ingresos_anual'] / 1e6,
                 color='#FF9800', alpha=0.7, label='ARR (eje izquierdo)')
ax2.plot(monthly_arr.index, monthly_arr['comision_pct'],
         color='#9C27B0', linewidth=2, label='Comisión media % (eje derecho)')

ax1.set_title('Ingresos Anuales Recurrentes (ARR) y Comisión Media', fontweight='bold', fontsize=13)
ax1.set_ylabel('ARR (Millones €, sin IVA)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax2.set_ylabel('Comisión media anual (%)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.3f}%'))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

## 8. Top días con mayores aportaciones

In [ ]:
top_aportaciones = df[df['aportaciones_dia'] > 0].nlargest(15, 'aportaciones_dia')[['aportaciones_dia', 'volumen']]
top_aportaciones.columns = ['Aportación neta (€)', 'Volumen total (€)']
top_aportaciones['Aportación neta (€)'] = top_aportaciones['Aportación neta (€)'].map('{:,.0f}'.format)
top_aportaciones['Volumen total (€)'] = top_aportaciones['Volumen total (€)'].map('{:,.0f}'.format)
print("Top 15 días con mayor aportación neta:")
print(top_aportaciones.to_string())

## 9. Tabla resumen anual completa

In [ ]:
resumen = anual[['aportaciones_netas', 'dias_con_aportacion', 'dias_con_retirada',
                  'aportacion_media_dia', 'aportacion_max_dia', 'volumen_fin',
                  'ingresos_anuales', 'crecimiento_volumen_pct']].copy()

resumen.columns = ['Aportaciones netas (€)', 'Días c/aportación', 'Días c/retirada',
                   'Aportación media/día (€)', 'Aportación máx/día (€)',
                   'Volumen cierre (€)', 'ARR (€)', 'Crecim. volumen (%)']

pd.set_option('display.float_format', '{:,.0f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(resumen.to_string())

## 10. Tasa de crecimiento anual compuesto (CAGR)

In [ ]:
años_totales = (df.index[-1] - df.index[0]).days / 365.25

cagr_volumen = (volumen_final / volumen_inicio) ** (1 / años_totales) - 1
cagr_arr = (df['ingresos_anual'].iloc[-1] / df['ingresos_anual'].iloc[0]) ** (1 / años_totales) - 1

# Aportaciones anuales por periodo para ver tendencia
aport_2016 = anual.loc[2016, 'aportaciones_netas'] if 2016 in anual.index else np.nan
aport_last  = anual.iloc[-2]['aportaciones_netas']  # último año completo
años_aport  = anual.iloc[-2].name - 2016
cagr_aport  = (aport_last / aport_2016) ** (1 / años_aport) - 1 if años_aport > 0 else np.nan

print(f"Periodo analizado: {años_totales:.1f} años")
print(f"CAGR Volumen gestionado:        {cagr_volumen*100:.1f}% anual")
print(f"CAGR Ingresos recurrentes (ARR): {cagr_arr*100:.1f}% anual")
print(f"CAGR Aportaciones anuales:       {cagr_aport*100:.1f}% anual (2016 → {anual.iloc[-2].name})")

## 11. Estacionalidad — últimos 5 años por serie (valor absoluto y % vs enero)

In [ ]:
meses_nombre = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

# ── Datos base ────────────────────────────────────────────────────────────────
mensual_pivot = (
    df.groupby(['año', 'mes'])['aportaciones_dia']
    .sum()
    .unstack(level='mes')
)

# Valor en el 1 de enero (primer registro disponible de cada año)
primer_dia_año = df.groupby('año').first()
volumen_1ene      = primer_dia_año['volumen']          # columna volumen a 1-ene
aport_acum_1ene   = primer_dia_año['aportaciones_acum'] # aportaciones acumuladas a 1-ene

# Últimos 5 años
ultimos_5     = sorted(mensual_pivot.index.unique())[-5:]
datos_5       = mensual_pivot.loc[ultimos_5]
vol_1ene_5    = volumen_1ene.reindex(ultimos_5)
aport_1ene_5  = aport_acum_1ene.reindex(ultimos_5)

colores_años = ['#E53935', '#FB8C00', '#43A047', '#1E88E5', '#8E24AA']
marcadores   = ['o', 's', '^', 'D', 'v']

fig = plt.figure(figsize=(18, 11))
ax_top = fig.add_subplot(2, 1, 1)
ax_bl  = fig.add_subplot(2, 2, 3)
ax_br  = fig.add_subplot(2, 2, 4)

def plot_series(ax, get_vals_fn, ylabel, title, hline=None):
    for i, año in enumerate(ultimos_5):
        fila       = datos_5.loc[año]
        meses_disp = fila.dropna().index.tolist()
        vals       = get_vals_fn(fila, año, meses_disp)
        if vals is None:
            continue
        etiquetas = [meses_nombre[m - 1] for m in meses_disp]
        ax.plot(etiquetas, vals,
                color=colores_años[i], marker=marcadores[i],
                linewidth=2.2, markersize=8, label=str(año))
    if hline is not None:
        ax.axhline(hline, color='grey', linewidth=0.9, linestyle='--')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('Mes')
    ax.set_ylabel(ylabel)
    ax.legend(title='Año', frameon=False, fontsize=9)
    ax.grid(axis='y', alpha=0.3)

# ── TOP: Valor absoluto ───────────────────────────────────────────────────────
def vals_absoluto(fila, año, meses_disp):
    return fila[meses_disp].values / 1e6

plot_series(ax_top, vals_absoluto,
            ylabel='Millones €',
            title='Aportaciones mensuales — valor absoluto (M€)',
            hline=0)
ax_top.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))

# ── BOTTOM LEFT: Aportaciones mes / Volumen 1-ene de ese año ─────────────────
def vals_pct_volumen(fila, año, meses_disp):
    base = vol_1ene_5.get(año, np.nan)
    if pd.isna(base) or base == 0:
        return None
    return (fila[meses_disp].values / base) * 100

plot_series(ax_bl, vals_pct_volumen,
            ylabel='% sobre volumen a 1 de enero',
            title='Aportaciones mensuales / Volumen 1-ene (%)',
            hline=0)
ax_bl.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.2f}%'))

# ── BOTTOM RIGHT: Aportaciones mes / Aportaciones acum. 1-ene de ese año ─────
def vals_pct_aport_acum(fila, año, meses_disp):
    base = aport_1ene_5.get(año, np.nan)
    if pd.isna(base) or base == 0:
        return None
    return (fila[meses_disp].values / base) * 100

plot_series(ax_br, vals_pct_aport_acum,
            ylabel='% sobre aportaciones acumuladas a 1 de enero',
            title='Aportaciones mensuales / Aport. acum. 1-ene (%)',
            hline=0)
ax_br.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.2f}%'))

plt.suptitle('Estacionalidad de aportaciones — últimos 5 años',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Tabla resumen
tabla = datos_5.copy() / 1e6
tabla.columns = [meses_nombre[m - 1] for m in tabla.columns]
tabla.index.name = 'Año'
print('\nAportaciones mensuales (M€):')
print(tabla.round(1).to_string())

---
## 12. Carga del PyG histórico (2021–2025)

In [ ]:
EXCEL_FILE = 'Datos financieros Indexa Capital Group.xlsx'

raw = pd.read_excel(EXCEL_FILE, sheet_name='PyG anual', header=None)

# Anos: fila 0, cols 1-5 son datetime objects
years = [dt.year for dt in raw.iloc[0, 1:].tolist()]

# Valores por indice de fila (ya son float, no hay formato europeo)
#  6 -> Comisiones percibidas
# 16 -> MARGEN BRUTO
# 17 -> Gastos de personal
# 18 -> Gastos generales
# 19 -> Amortizacion
# 24 -> RESULTADO ACTIVIDAD EXPLOTACION
# 33 -> Impuesto sobre beneficios
# 36 -> RESULTADO CONSOLIDADO
ROW_MAP = {
    'comisiones':    6,
    'margen_bruto':  16,
    'personal':      17,
    'generales':     18,
    'amortizacion':  19,
    'ebit':          24,
    'impuestos':     33,
    'resultado_neto':36,
}

hist = pd.DataFrame(index=years)
hist.index.name = 'ano'
for key, row_idx in ROW_MAP.items():
    hist[key] = raw.iloc[row_idx, 1:].astype(float).values

print('PyG historico cargado correctamente:')
print(hist.applymap(lambda x: f'{x:>12,.0f} EUR').to_string())


## 13. Ratios históricos y métricas operativas

In [ ]:
# ── Ratios sobre ingresos (comisiones) ───────────────────────────────────────
ratios = pd.DataFrame(index=hist.index)
ratios['personal_pct']    = hist['personal'].abs()    / hist['comisiones'] * 100
ratios['generales_pct']   = hist['generales'].abs()   / hist['comisiones'] * 100
ratios['amort_pct']       = hist['amortizacion'].abs()/ hist['comisiones'] * 100
ratios['margen_ebit_pct'] = hist['ebit']              / hist['comisiones'] * 100
ratios['margen_neto_pct'] = hist['resultado_neto']    / hist['comisiones'] * 100
ratios['tipo_efectivo']   = hist['impuestos'].abs()   / hist['ebit'].replace(0, np.nan) * 100

# ── Volumen gestionado a cierre de año (desde CSV) ───────────────────────────
vol_cierre = df.groupby('año')['volumen'].last()
arr_cierre = df.groupby('año')['ingresos_anual'].last()

ratios['volumen_MM']   = vol_cierre.reindex(hist.index) / 1e6
ratios['arr_MM']       = arr_cierre.reindex(hist.index) / 1e6
ratios['comision_pct_vol'] = hist['comisiones'] / (vol_cierre.reindex(hist.index)) * 100

print("Ratios históricos (%  sobre comisiones salvo indicado):\n")
print(ratios.round(1).to_string())

# ── Gráfico ratios de costes ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(hist.index, ratios['personal_pct'],  marker='o', color='#E53935', label='Personal')
axes[0].plot(hist.index, ratios['generales_pct'], marker='s', color='#1E88E5', label='Generales')
axes[0].set_title('Costes / Comisiones (%)', fontweight='bold')
axes[0].set_ylabel('%'); axes[0].legend(frameon=False); axes[0].grid(alpha=0.3)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))

axes[1].plot(hist.index, ratios['margen_ebit_pct'], marker='o', color='#43A047', label='Margen EBIT')
axes[1].plot(hist.index, ratios['margen_neto_pct'], marker='s', color='#8E24AA', label='Margen neto')
axes[1].set_title('Márgenes sobre comisiones (%)', fontweight='bold')
axes[1].set_ylabel('%'); axes[1].legend(frameon=False); axes[1].grid(alpha=0.3)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))

axes[2].bar(hist.index.astype(str), hist['comisiones']/1e6, color='#FF9800', label='Comisiones')
axes[2].plot(hist.index.astype(str), hist['resultado_neto']/1e6, marker='o', color='#8E24AA',
             linewidth=2, label='Resultado neto')
axes[2].set_title('Comisiones vs Resultado neto (M€)', fontweight='bold')
axes[2].set_ylabel('M€'); axes[2].legend(frameon=False); axes[2].grid(alpha=0.3, axis='y')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.1f}M'))

plt.tight_layout()
plt.show()

## 14. Proyecciones 2026E – 2030E — Metodología explicada

### ¿Cómo se obtiene cada valor proyectado?

#### 📈 Ingresos (Comisiones percibidas)
La fórmula es: **Volumen gestionado × Comisión media anual**

| Paso | Detalle |
|------|---------|
| **Base** | Volumen cierre 2025 = **4.411 M€** (último valor del CSV) |
| **Crecimiento** | Se aplica un CAGR del **28%** anual (editable). El histórico real 2021-2025 es ~35%, se aplica un descuento por madurez |
| **Comisión** | **0,245%** anual sobre el volumen (actual según CSV) |
| **Ancla 2026** | Si el ARR real del CSV (12,18 M€ en abr-2026) supera la proyección pura, se usa ese valor |

Ejemplo 2027: 4.411B × 1,28² × 0,245% = **17,7 M€**

---

#### 👥 Gastos de personal
Se usa **regresión lineal** sobre el ratio histórico *personal / comisiones*:

| Año | Ratio real |
|-----|-----------|
| 2021 | 35,8% |
| 2022 | 36,0% |
| 2023 | 36,1% |
| 2024 | 36,3% |
| 2025 | **29,2%** ← gran salto por apalancamiento operativo |

La regresión calcula la tendencia: **cada año el ratio baja ~1,3 pp**.  
Proyectado = comisiones_año × ratio_proyectado  
Ejemplo 2027: ratio proyectado ≈ 29,6% → 17,7M × 29,6% = **5,2 M€**

---

#### 🏢 Gastos generales
Mismo método. Ratio histórico *generales / comisiones*:

| Año | Ratio real |
|-----|-----------|
| 2021 | 42,4% |
| 2022 | 42,4% |
| 2023 | 42,6% |
| 2024 | 43,8% |
| 2025 | **33,7%** |

Tendencia regresión: **cada año el ratio baja ~1,6 pp**  
Ejemplo 2027: ratio proyectado ≈ 34,6% → 17,7M × 34,6% = **6,1 M€**

---

#### 📉 Amortización
Regresión en valor absoluto (histórico: 161K→153K→101K→67K→63K, claramente decreciente).  
Se aplica un **suelo de 40.000 €** para evitar valores negativos — refleja que activos futuros seguirán amortizándose.

---

#### 💰 EBIT y Resultado neto
- **EBIT** = Comisiones + Personal + Generales + Amortización (todos los gastos son negativos)
- **Impuestos** = EBIT × 27% (tipo efectivo real de 2023-2025, editable)
- **Resultado neto** = EBIT − Impuestos

> ⚠️ **Limitaciones**: modelo lineal, no incluye posibles cambios regulatorios, nuevos productos, variaciones de mercado que afecten al volumen, ni contrataciones extraordinarias. Úsalo como orden de magnitud, no como previsión exacta.

In [ ]:
from sklearn.linear_model import LinearRegression

PROJ_YEARS = [2026, 2027, 2028, 2029, 2030]

# ════════════════════════════════════════════════════════════════════
# SUPUESTOS EDITABLES
# ════════════════════════════════════════════════════════════════════
ARR_2026_ANCHOR   = 12_184_300  # ARR a abr-2026 del CSV (EUR)        # <- EDITAR
CAGR_VOL_FWD      = 0.28        # Crecimiento anual volumen 2026-30    # <- EDITAR
COMISION_ANUAL    = 0.00245     # Comision media proyectada            # <- EDITAR
TIPO_IMPOSITIVO   = 0.27        # Tipo efectivo IS                     # <- EDITAR
AMORT_SUELO       = 40_000      # Minimo amortizacion anual (EUR)      # <- EDITAR
# ════════════════════════════════════════════════════════════════════

# ── Proyeccion de ingresos ────────────────────────────────────────────────────
vol_2025 = df[df['año'] == 2025]['volumen'].iloc[-1]

vol_proj, com_proj = {}, {}
for i, yr in enumerate(PROJ_YEARS):
    vol_proj[yr] = vol_2025 * (1 + CAGR_VOL_FWD) ** (i + 1)
    com_proj[yr] = vol_proj[yr] * COMISION_ANUAL

com_proj[2026] = max(com_proj[2026], ARR_2026_ANCHOR)

# ── Regresion lineal de ratios de costes ──────────────────────────────────────
X = np.array(hist.index).reshape(-1, 1)

personal_ratios  = hist['personal'].abs()  / hist['comisiones']
generales_ratios = hist['generales'].abs() / hist['comisiones']
amort_abs        = hist['amortizacion'].abs()

reg_p = LinearRegression().fit(X, personal_ratios.values)
reg_g = LinearRegression().fit(X, generales_ratios.values)
reg_a = LinearRegression().fit(X, amort_abs.values)

X_fut = np.array(PROJ_YEARS).reshape(-1, 1)
personal_proj  = np.clip(reg_p.predict(X_fut), 0, None)
generales_proj = np.clip(reg_g.predict(X_fut), 0, None)
amort_proj     = np.clip(reg_a.predict(X_fut), AMORT_SUELO, None)  # suelo 40K

# ── Construir PyG proyectado ──────────────────────────────────────────────────
proj = pd.DataFrame(index=PROJ_YEARS)
proj.index.name = 'año'

proj['comisiones']    = [com_proj[y] for y in PROJ_YEARS]
proj['personal']      = -proj['comisiones'] * personal_proj
proj['generales']     = -proj['comisiones'] * generales_proj
proj['amortizacion']  = -amort_proj
proj['ebit']          = proj[['comisiones','personal','generales','amortizacion']].sum(axis=1)
proj['impuestos']     = -proj['ebit'].clip(lower=0) * TIPO_IMPOSITIVO
proj['resultado_neto']= proj['ebit'] + proj['impuestos']
proj['margen_ebit_pct'] = proj['ebit']           / proj['comisiones'] * 100
proj['margen_neto_pct'] = proj['resultado_neto'] / proj['comisiones'] * 100

# ── Tabla combinada historico + proyecciones ──────────────────────────────────
combinado = pd.concat([
    hist[['comisiones','personal','generales','amortizacion','ebit','impuestos','resultado_neto']],
    proj[['comisiones','personal','generales','amortizacion','ebit','impuestos','resultado_neto']]
])
all_labels = [str(y) for y in hist.index] + [f"{y}E" for y in PROJ_YEARS]

# Ratios combinados
all_margen_ebit = list(ratios['margen_ebit_pct']) + list(proj['margen_ebit_pct'])
all_margen_neto = list(ratios['margen_neto_pct']) + list(proj['margen_neto_pct'])

sep = "=" * 115
print(sep)
print(f"{'':32s}" + "".join(f"{l:>13s}" for l in all_labels))
print(sep)
filas = [
    ('Comisiones (M€)',    'comisiones',     1e6),
    ('  Gastos personal',  'personal',       1e6),
    ('  Gastos generales', 'generales',      1e6),
    ('  Amortizacion',     'amortizacion',   1e6),
    ('EBIT (M€)',          'ebit',           1e6),
    ('  Impuestos',        'impuestos',      1e6),
    ('Resultado neto (M€)','resultado_neto', 1e6),
]
for label, col, div in filas:
    if col in ('ebit', 'resultado_neto'):
        print("-" * 115)
    vals = "".join(f"{v/div:>13.2f}" for v in combinado[col])
    print(f"{label:32s}{vals}")
print("=" * 115)
print(f"{'Margen EBIT %':32s}" + "".join(f"{v:>12.1f}%" for v in all_margen_ebit))
print(f"{'Margen neto %':32s}" + "".join(f"{v:>12.1f}%" for v in all_margen_neto))
print(f"{'Volumen gestionado (B€)':32s}" +
      "".join(f"{vol_cierre.get(y, vol_proj.get(y,0))/1e9:>13.2f}"
              for y in list(hist.index) + PROJ_YEARS))
print(sep)

In [ ]:
# ── Grafico historico + proyecciones hasta 2030 ──────────────────────────────
all_com    = list(hist['comisiones'])     + list(proj['comisiones'])
all_neto   = list(hist['resultado_neto']) + list(proj['resultado_neto'])
all_ebit   = list(hist['ebit'])           + list(proj['ebit'])
all_labels = [str(y) for y in hist.index] + [f"{y}E" for y in PROJ_YEARS]
all_margen_ebit = list(ratios['margen_ebit_pct']) + list(proj['margen_ebit_pct'])
all_margen_neto = list(ratios['margen_neto_pct']) + list(proj['margen_neto_pct'])

c_hist = '#90A4AE'
c_proj = '#1E88E5'
colors_bar = [c_hist] * len(hist) + [c_proj] * len(PROJ_YEARS)
colors_net = ['#43A047' if v >= 0 else '#E53935' for v in all_neto]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
x = np.arange(len(all_labels))
w = 0.35

# Panel izquierdo: barras comisiones + resultado neto
axes[0].bar(x - w/2, [v/1e6 for v in all_com],  w, color=colors_bar, alpha=0.9, label='Comisiones')
axes[0].bar(x + w/2, [v/1e6 for v in all_neto], w, color=colors_net,  alpha=0.9, label='Resultado neto')
axes[0].set_xticks(x); axes[0].set_xticklabels(all_labels, rotation=30)
axes[0].set_title('Comisiones vs Resultado neto (M€)', fontweight='bold')
axes[0].set_ylabel('M€')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}M'))
axes[0].axvline(len(hist) - 0.5, color='black', linewidth=1, linestyle=':', alpha=0.5)
axes[0].text(len(hist) - 0.4, max(all_com)/1e6 * 0.97, 'Proyeccion →', fontsize=9, color='grey')
for xi, (c, n) in enumerate(zip(all_com, all_neto)):
    axes[0].text(xi - w/2, c/1e6 + 0.2, f'{c/1e6:.1f}M', ha='center', fontsize=7, color='#455A64')
    axes[0].text(xi + w/2, n/1e6 + 0.2, f'{n/1e6:.1f}M', ha='center', fontsize=7,
                 color='#2E7D32' if n >= 0 else '#C62828')
axes[0].legend(frameon=False); axes[0].grid(axis='y', alpha=0.3)

# Panel derecho: margenes
axes[1].plot(all_labels, all_margen_ebit, marker='o', color='#43A047', linewidth=2, label='Margen EBIT')
axes[1].plot(all_labels, all_margen_neto, marker='s', color='#8E24AA', linewidth=2, label='Margen neto')
# Tramo proyectado en discontinuo
cut = len(hist) - 1
axes[1].plot(all_labels[cut:cut+2], all_margen_ebit[cut:cut+2], color='#43A047', linewidth=2, linestyle='--')
axes[1].plot(all_labels[cut:cut+2], all_margen_neto[cut:cut+2], color='#8E24AA', linewidth=2, linestyle='--')
for i in range(len(hist), len(all_labels)):
    axes[1].plot(all_labels[i-1:i+1], all_margen_ebit[i-1:i+1], color='#43A047', linewidth=2, linestyle='--')
    axes[1].plot(all_labels[i-1:i+1], all_margen_neto[i-1:i+1], color='#8E24AA', linewidth=2, linestyle='--')
axes[1].axvline(len(hist) - 0.5, color='black', linewidth=1, linestyle=':', alpha=0.5)
axes[1].set_title('Evolucion de margenes (%)', fontweight='bold')
axes[1].set_ylabel('%')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}%'))
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(frameon=False); axes[1].grid(alpha=0.3)

plt.suptitle('Indexa Capital — PyG historico (2021-2025) y proyecciones (2026E-2030E)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 15. Proyecciones mejoradas - modelo de drivers del informe

Este bloque mantiene el modelo anterior como referencia, pero introduce una proyeccion bottom-up mas cercana a la logica del PDF.

### Que cambia

1. Clientes: las altas netas salen de `marketing / CAC` y se resta `churn`.
2. AUM: se construye con aportacion inicial, aportacion recurrente e impacto mercado.
3. Comisiones: se calculan sobre AUM medio anual, no sobre AUM de cierre.
4. Costes: se separa marketing del resto de generales y se usan drivers explicitos.
5. Benchmark: se compara cada ano con el escenario central del informe.


In [ ]:
PROJ_YEARS = [2026, 2027, 2028, 2029, 2030]

# --- Anclas vivas del notebook -------------------------------------------------
AUM_2025 = float(vol_cierre.loc[2025])
AUM_2026_LIVE = float(df['volumen'].iloc[-1])
ARR_2026_LIVE = float(df['ingresos_anual'].iloc[-1])

# --- Supuestos operativos inspirados en el informe -----------------------------
CLIENTS_2025 = 134_000
CLIENTS_2026_LIVE = 148_621

CHURN_ANNUAL = 0.056
INITIAL_TICKET = 9_400
MONTHLY_NET_CONTRIB = 438
MARKET_RETURN = 0.05
FEE_RATE = 0.00250

MARKETING_2025 = 1_600_000
MARKETING_GROWTH = 0.25
CAC_2026 = 45
CAC_GROWTH = 0.05

PERSONNEL_2025 = abs(float(hist.loc[2025, 'personal']))
PERSONNEL_GROWTH = 0.11

GENERALS_2025 = abs(float(hist.loc[2025, 'generales']))
OTHER_GENERALS_2025 = max(GENERALS_2025 - MARKETING_2025, 0.0)
OTHER_GENERALS_GROWTH = 0.10

AMORT_2026 = 60_000
AMORT_GROWTH = 0.03

TAX_RATE = {2026: 0.25, 2027: 0.23, 2028: 0.23, 2029: 0.23, 2030: 0.23}
CASH_CONVERSION = {2026: 0.773, 2027: 0.832, 2028: 0.851, 2029: 0.866, 2030: 0.877}
NET_CASH_2025 = 2_157_000

# --- Escenario central del PDF para comparar ----------------------------------
bench_pdf = pd.DataFrame(
    {
        'clientes': [178_000, 230_000, 292_000, 366_000, 454_000],
        'aum_fin': [5_816e6, 7_627e6, 9_922e6, 12_803e6, 16_388e6],
        'comisiones': [13_494e3, 17_738e3, 23_152e3, 29_976e3, 38_496e3],
        'ebit': [6_484e3, 9_840e3, 14_171e3, 19_848e3, 27_007e3],
        'resultado_neto': [4_863e3, 7_577e3, 10_911e3, 15_283e3, 20_795e3],
        'tesoreria_neta': [5_917e3, 12_220e3, 21_508e3, 34_744e3, 52_983e3],
    },
    index=PROJ_YEARS,
)
bench_pdf.index.name = 'ano'

# --- Proyeccion bottom-up ------------------------------------------------------
rows = []
aum_prev = AUM_2025
clients_prev = CLIENTS_2025
net_cash_prev = NET_CASH_2025

for year in PROJ_YEARS:
    marketing = MARKETING_2025 * (1 + MARKETING_GROWTH) ** (year - 2025)
    cac = CAC_2026 * (1 + CAC_GROWTH) ** (year - 2026)
    net_adds = marketing / cac

    clients_end = clients_prev * (1 - CHURN_ANNUAL) + net_adds
    if year == 2026:
        clients_end = max(clients_end, CLIENTS_2026_LIVE)
    avg_clients = (clients_prev + clients_end) / 2

    initial_flows = net_adds * INITIAL_TICKET
    recurring_flows = avg_clients * MONTHLY_NET_CONTRIB * 12
    total_net_flows = initial_flows + recurring_flows

    market_impact = (aum_prev + 0.5 * total_net_flows) * MARKET_RETURN
    aum_end = aum_prev + total_net_flows + market_impact
    if year == 2026:
        aum_end = max(aum_end, AUM_2026_LIVE)
    avg_aum = 0.5 * (aum_prev + aum_end)

    fees = avg_aum * FEE_RATE
    if year == 2026:
        fees = max(fees, ARR_2026_LIVE)

    personnel = PERSONNEL_2025 * (1 + PERSONNEL_GROWTH) ** (year - 2025)
    other_generals = OTHER_GENERALS_2025 * (1 + OTHER_GENERALS_GROWTH) ** (year - 2025)
    amort = AMORT_2026 * (1 + AMORT_GROWTH) ** (year - 2026)
    total_generals = marketing + other_generals

    ebit = fees - personnel - total_generals - amort
    taxes = -max(ebit, 0) * TAX_RATE[year]
    net_profit = ebit + taxes

    fcf = max(net_profit, 0) * CASH_CONVERSION[year]
    net_cash = net_cash_prev + fcf

    rows.append(
        {
            'ano': year,
            'clientes': clients_end,
            'altas_netas': net_adds,
            'aum_inicio': aum_prev,
            'aum_fin': aum_end,
            'aum_medio': avg_aum,
            'aport_iniciales': initial_flows,
            'aport_recurrentes': recurring_flows,
            'impacto_mercado': market_impact,
            'comisiones': fees,
            'gasto_personal': personnel,
            'gasto_marketing': marketing,
            'gasto_generales_otros': other_generals,
            'gasto_amortizacion': amort,
            'ebit': ebit,
            'impuestos': taxes,
            'resultado_neto': net_profit,
            'cash_conversion': CASH_CONVERSION[year],
            'fcf': fcf,
            'tesoreria_neta': net_cash,
        }
    )

    aum_prev = aum_end
    clients_prev = clients_end
    net_cash_prev = net_cash

proj = pd.DataFrame(rows).set_index('ano')
proj['personal'] = -proj['gasto_personal']
proj['marketing'] = -proj['gasto_marketing']
proj['generales_otros'] = -proj['gasto_generales_otros']
proj['generales'] = -(proj['gasto_marketing'] + proj['gasto_generales_otros'])
proj['amortizacion'] = -proj['gasto_amortizacion']
proj['margen_ebit_pct'] = proj['ebit'] / proj['comisiones'] * 100
proj['margen_neto_pct'] = proj['resultado_neto'] / proj['comisiones'] * 100

# --- Tablas de salida ----------------------------------------------------------
drivers_view = pd.DataFrame(
    {
        'Clientes (k)': proj['clientes'] / 1e3,
        'Altas netas (k)': proj['altas_netas'] / 1e3,
        'AUM fin (B EUR)': proj['aum_fin'] / 1e9,
        'Aport. iniciales (M EUR)': proj['aport_iniciales'] / 1e6,
        'Aport. recurrentes (M EUR)': proj['aport_recurrentes'] / 1e6,
        'Impacto mercado (M EUR)': proj['impacto_mercado'] / 1e6,
        'Comisiones (M EUR)': proj['comisiones'] / 1e6,
    }
)

compare_view = pd.DataFrame(
    {
        'Clientes modelo (k)': proj['clientes'] / 1e3,
        'Clientes PDF (k)': bench_pdf['clientes'] / 1e3,
        'AUM modelo (B EUR)': proj['aum_fin'] / 1e9,
        'AUM PDF (B EUR)': bench_pdf['aum_fin'] / 1e9,
        'Comisiones modelo (M EUR)': proj['comisiones'] / 1e6,
        'Comisiones PDF (M EUR)': bench_pdf['comisiones'] / 1e6,
        'EBIT modelo (M EUR)': proj['ebit'] / 1e6,
        'EBIT PDF (M EUR)': bench_pdf['ebit'] / 1e6,
        'Neto modelo (M EUR)': proj['resultado_neto'] / 1e6,
        'Neto PDF (M EUR)': bench_pdf['resultado_neto'] / 1e6,
        'Caja modelo (M EUR)': proj['tesoreria_neta'] / 1e6,
        'Caja PDF (M EUR)': bench_pdf['tesoreria_neta'] / 1e6,
    }
)
compare_view['Delta AUM %'] = (compare_view['AUM modelo (B EUR)'] / compare_view['AUM PDF (B EUR)'] - 1) * 100
compare_view['Delta comisiones %'] = (
    compare_view['Comisiones modelo (M EUR)'] / compare_view['Comisiones PDF (M EUR)'] - 1
) * 100
compare_view['Delta neto %'] = (
    compare_view['Neto modelo (M EUR)'] / compare_view['Neto PDF (M EUR)'] - 1
) * 100

print('Supuestos principales del escenario base:')
print(
    pd.Series(
        {
            'AUM cierre 2025 (B EUR)': AUM_2025 / 1e9,
            'AUM vivo abr-2026 (B EUR)': AUM_2026_LIVE / 1e9,
            'ARR vivo abr-2026 (M EUR)': ARR_2026_LIVE / 1e6,
            'Clientes cierre 2025 (k)': CLIENTS_2025 / 1e3,
            'Clientes vivos 2026 (k)': CLIENTS_2026_LIVE / 1e3,
            'Churn anual (%)': CHURN_ANNUAL * 100,
            'Ticket inicial (EUR)': INITIAL_TICKET,
            'Aportacion recurrente anual (EUR)': MONTHLY_NET_CONTRIB * 12,
            'Retorno mercado (%)': MARKET_RETURN * 100,
            'Fee media (%)': FEE_RATE * 100,
            'Marketing 2025 (M EUR)': MARKETING_2025 / 1e6,
            'Marketing growth (%)': MARKETING_GROWTH * 100,
            'CAC 2026 (EUR)': CAC_2026,
            'Personal growth (%)': PERSONNEL_GROWTH * 100,
            'Generales ex mkt growth (%)': OTHER_GENERALS_GROWTH * 100,
        }
    ).round(2).to_string()
)

print('\nDrivers proyectados:')
print(drivers_view.round(2).to_string())

print('\nComparacion contra el escenario central del PDF:')
print(compare_view.round(2).to_string())

combinado = pd.concat(
    [
        hist[['comisiones', 'personal', 'generales', 'amortizacion', 'ebit', 'impuestos', 'resultado_neto']],
        proj[['comisiones', 'personal', 'generales', 'amortizacion', 'ebit', 'impuestos', 'resultado_neto']],
    ]
)
all_labels = [str(y) for y in hist.index] + [f'{y}E' for y in PROJ_YEARS]
all_margen_ebit = list(ratios['margen_ebit_pct']) + list(proj['margen_ebit_pct'])
all_margen_neto = list(ratios['margen_neto_pct']) + list(proj['margen_neto_pct'])

sep = '=' * 128
print('\n' + sep)
print(f"{'':32s}" + ''.join(f'{label:>12s}' for label in all_labels))
print(sep)
for label, col in [
    ('Comisiones (M EUR)', 'comisiones'),
    ('  Gastos personal', 'personal'),
    ('  Gastos generales', 'generales'),
    ('  Amortizacion', 'amortizacion'),
    ('EBIT (M EUR)', 'ebit'),
    ('  Impuestos', 'impuestos'),
    ('Resultado neto (M EUR)', 'resultado_neto'),
]:
    if col in ('ebit', 'resultado_neto'):
        print('-' * 128)
    values = ''.join(f'{v / 1e6:>12.2f}' for v in combinado[col])
    print(f'{label:32s}{values}')
print('=' * 128)
print(f"{'Margen EBIT %':32s}" + ''.join(f'{v:>11.1f}%' for v in all_margen_ebit))
print(f"{'Margen neto %':32s}" + ''.join(f'{v:>11.1f}%' for v in all_margen_neto))
print(
    f"{'AUM cierre (B EUR)':32s}"
    + ''.join(
        f'{(vol_cierre.loc[y] if y in hist.index else proj.loc[y, "aum_fin"]) / 1e9:>12.2f}'
        for y in list(hist.index) + PROJ_YEARS
    )
)
print(sep)


In [ ]:
# --- Graficos: modelo vs benchmark del informe --------------------------------
all_com = list(hist['comisiones']) + list(proj['comisiones'])
all_neto = list(hist['resultado_neto']) + list(proj['resultado_neto'])
all_labels = [str(y) for y in hist.index] + [f'{y}E' for y in PROJ_YEARS]
x = np.arange(len(all_labels))
proj_x = x[-len(PROJ_YEARS):]

c_hist = '#90A4AE'
c_proj = '#1E88E5'
colors_bar = [c_hist] * len(hist) + [c_proj] * len(PROJ_YEARS)
colors_net = ['#43A047' if v >= 0 else '#E53935' for v in all_neto]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
w = 0.35

# Panel izquierdo: historico + proyeccion propia con benchmark del PDF encima
axes[0].bar(x - w / 2, [v / 1e6 for v in all_com], w, color=colors_bar, alpha=0.90, label='Comisiones modelo')
axes[0].bar(x + w / 2, [v / 1e6 for v in all_neto], w, color=colors_net, alpha=0.85, label='Resultado neto modelo')
axes[0].plot(proj_x - w / 2, bench_pdf['comisiones'] / 1e6, color='#0D47A1', marker='o', linestyle='--', linewidth=2, label='Comisiones PDF')
axes[0].plot(proj_x + w / 2, bench_pdf['resultado_neto'] / 1e6, color='#1B5E20', marker='s', linestyle='--', linewidth=2, label='Resultado neto PDF')
axes[0].set_xticks(x)
axes[0].set_xticklabels(all_labels, rotation=30)
axes[0].set_title('PyG historico y proyeccion comparada', fontweight='bold')
axes[0].set_ylabel('M EUR')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}M'))
axes[0].axvline(len(hist) - 0.5, color='black', linewidth=1, linestyle=':', alpha=0.5)
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend(frameon=False, fontsize=8)

# Panel derecho: AUM y clientes, modelo vs PDF
line_model_aum = axes[1].plot(PROJ_YEARS, proj['aum_fin'] / 1e9, color='#1565C0', marker='o', linewidth=2.5, label='AUM modelo')[0]
line_pdf_aum = axes[1].plot(PROJ_YEARS, bench_pdf['aum_fin'] / 1e9, color='#1565C0', marker='o', linestyle='--', linewidth=2, label='AUM PDF')[0]
axes[1].set_title('Drivers de crecimiento: AUM y clientes', fontweight='bold')
axes[1].set_ylabel('AUM (B EUR)', color='#1565C0')
axes[1].tick_params(axis='y', labelcolor='#1565C0')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}B'))
axes[1].set_xticks(PROJ_YEARS)
axes[1].grid(alpha=0.3)

ax2 = axes[1].twinx()
line_model_clients = ax2.plot(PROJ_YEARS, proj['clientes'] / 1e3, color='#E65100', marker='s', linewidth=2.5, label='Clientes modelo')[0]
line_pdf_clients = ax2.plot(PROJ_YEARS, bench_pdf['clientes'] / 1e3, color='#E65100', marker='s', linestyle='--', linewidth=2, label='Clientes PDF')[0]
ax2.set_ylabel('Clientes (miles)', color='#E65100')
ax2.tick_params(axis='y', labelcolor='#E65100')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}k'))

lines = [line_model_aum, line_pdf_aum, line_model_clients, line_pdf_clients]
axes[1].legend(lines, [line.get_label() for line in lines], frameon=False, fontsize=8, loc='upper left')

plt.suptitle('Indexa Capital - modelo bottom-up vs escenario central del informe', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 16. Escenarios de AUM y beneficio neto hasta 2030

Aqui forzamos tres trayectorias simples de crecimiento del AUM desde el dato vivo de 2026 y observamos el impacto en comisiones y beneficio neto.

Supuesto importante: solo cambia el crecimiento del AUM. La estructura de costes se mantiene igual que en el escenario base del bloque 15 para visualizar el apalancamiento operativo.


In [ ]:
SCENARIO_GROWTH = {
    'Pesimista 20%': 0.20,
    'Neutral 35%': 0.35,
    'Optimista 50%': 0.50,
}
SCENARIO_COLORS = {
    'Pesimista 20%': '#2E7D32',
    'Neutral 35%': '#1565C0',
    'Optimista 50%': '#EF6C00',
}

AUM_BASE_SCEN = float(df['volumen'].iloc[-1])
YEARS_SCEN = [2026, 2027, 2028, 2029, 2030]

scenario_rows = []
for scenario_name, growth in SCENARIO_GROWTH.items():
    aum_prev = AUM_BASE_SCEN
    for year in YEARS_SCEN:
        if year == 2026:
            aum_end = AUM_BASE_SCEN
        else:
            aum_end = aum_prev * (1 + growth)
        avg_aum = 0.5 * (aum_prev + aum_end)
        fees = max(avg_aum * FEE_RATE, ARR_2026_LIVE) if year == 2026 else avg_aum * FEE_RATE

        personnel = PERSONNEL_2025 * (1 + PERSONNEL_GROWTH) ** (year - 2025)
        marketing = MARKETING_2025 * (1 + MARKETING_GROWTH) ** (year - 2025)
        other_generals = OTHER_GENERALS_2025 * (1 + OTHER_GENERALS_GROWTH) ** (year - 2025)
        amort = AMORT_2026 * (1 + AMORT_GROWTH) ** (year - 2026)

        ebit = fees - personnel - marketing - other_generals - amort
        taxes = -max(ebit, 0) * TAX_RATE[year]
        net_profit = ebit + taxes

        scenario_rows.append(
            {
                'escenario': scenario_name,
                'ano': year,
                'aum_fin': aum_end,
                'aum_medio': avg_aum,
                'comisiones': fees,
                'ebit': ebit,
                'resultado_neto': net_profit,
            }
        )
        aum_prev = aum_end

scenario_df = pd.DataFrame(scenario_rows)

summary_2030 = (
    scenario_df[scenario_df['ano'] == 2030]
    .set_index('escenario')[['aum_fin', 'comisiones', 'ebit', 'resultado_neto']]
    .rename(
        columns={
            'aum_fin': 'AUM 2030E (B EUR)',
            'comisiones': 'Comisiones 2030E (M EUR)',
            'ebit': 'EBIT 2030E (M EUR)',
            'resultado_neto': 'Beneficio neto 2030E (M EUR)',
        }
    )
)
summary_2030['AUM 2030E (B EUR)'] /= 1e9
summary_2030['Comisiones 2030E (M EUR)'] /= 1e6
summary_2030['EBIT 2030E (M EUR)'] /= 1e6
summary_2030['Beneficio neto 2030E (M EUR)'] /= 1e6

print('Resumen 2030E por escenario:')
print(summary_2030.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for scenario_name in SCENARIO_GROWTH:
    temp = scenario_df[scenario_df['escenario'] == scenario_name]
    color = SCENARIO_COLORS[scenario_name]
    axes[0].plot(temp['ano'], temp['aum_fin'] / 1e9, marker='o', linewidth=2.5, color=color, label=scenario_name)
    axes[1].plot(temp['ano'], temp['resultado_neto'] / 1e6, marker='o', linewidth=2.5, color=color, label=scenario_name)

axes[0].axhline(bench_pdf.loc[2030, 'aum_fin'] / 1e9, color='#455A64', linestyle='--', linewidth=1.5, alpha=0.9)
axes[0].text(2026.05, bench_pdf.loc[2030, 'aum_fin'] / 1e9 + 0.2, 'PDF central 2030: 16.4B', color='#455A64', fontsize=9)
axes[0].set_title('Escenarios de AUM hasta 2030', fontweight='bold')
axes[0].set_ylabel('AUM (B EUR)')
axes[0].set_xticks(YEARS_SCEN)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}B'))
axes[0].grid(alpha=0.3)
axes[0].legend(frameon=False)

axes[1].axhline(0, color='#90A4AE', linewidth=1)
axes[1].axhline(bench_pdf.loc[2030, 'resultado_neto'] / 1e6, color='#455A64', linestyle='--', linewidth=1.5, alpha=0.9)
axes[1].text(2026.05, bench_pdf.loc[2030, 'resultado_neto'] / 1e6 + 0.4, 'PDF central 2030: 20.8M', color='#455A64', fontsize=9)
axes[1].set_title('Impacto en beneficio neto', fontweight='bold')
axes[1].set_ylabel('Beneficio neto (M EUR)')
axes[1].set_xticks(YEARS_SCEN)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}M'))
axes[1].grid(alpha=0.3)

for scenario_name in SCENARIO_GROWTH:
    temp = scenario_df[(scenario_df['escenario'] == scenario_name) & (scenario_df['ano'] == 2030)]
    color = SCENARIO_COLORS[scenario_name]
    aum_2030 = temp['aum_fin'].iloc[0] / 1e9
    net_2030 = temp['resultado_neto'].iloc[0] / 1e6
    axes[0].text(2030.04, aum_2030, f'{aum_2030:.1f}B', color=color, fontsize=10, fontweight='bold')
    axes[1].text(2030.04, net_2030, f'{net_2030:.1f}M', color=color, fontsize=10, fontweight='bold')

plt.suptitle('AUM y beneficio neto bajo tres escenarios de crecimiento', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
